# Commercial Intelligence: Exploratory Data Analysis (EDA)

This notebook explores the cleaned transaction data to identify patterns in product performance, seasonality, and customer behavior. These insights form the foundation for our **Pricing** and **Demand Forecasting** models.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Load cleaned data
df = pd.read_csv('../data/processed/retail_clean.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Set display options
pd.options.display.float_format = '{:,.2f}'.format
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,DayOfWeek,Week
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,83.40,2009,12,1,49
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00,2009,12,1,49
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,81.00,2009,12,1,49
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,100.80,2009,12,1,49
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,30.00,2009,12,1,49


## 1. Top 20 Products by Revenue
Identifying our "Hero" products that drive the bulk of commercial value.

In [2]:
import plotly.express as px
admin_codes = ['M', 'POST', 'DOT', 'ADJUST', 'ADJUST2', 'BANK CHARGES', 'PADS', 'gift_0001_80343', 'B', 'CRUK']
top_20 = df[~df['Category'].isnull() & ~df['StockCode'].isin(admin_codes)].groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(20).reset_index()

fig = px.bar(
    top_20, 
    x='Revenue', 
    y='Description', 
    orientation='h',
    title='Top 20 Products by Total Revenue',
    labels={'Revenue': 'Revenue (£)', 'Description': 'Product Name'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**Commercial Insight:**
The top products represent your core "margin engines." A small percentage of SKUs often drive a disproportionate share of revenue (Pareto Principle). For these 20 products, even a 1% price optimization or a reduction in stock-outs will have a massive impact on the bottom line. These should be the first candidates for the **Pricing Intelligence** model.

## 2. Monthly Revenue Trend (2009 - 2011)
Visualizing growth and seasonality to inform supply chain planning.

In [3]:
monthly_revenue = df.set_index(pd.to_datetime(df['InvoiceDate'])).resample('ME')['Revenue'].sum().reset_index()
fig = px.line(
    monthly_revenue, 
    x='InvoiceDate', 
    y='Revenue', 
    title='Monthly Revenue Trend (2009-2011)',
    markers=True
)

# Add annotations for peaks and partial month
fig.add_annotation(x="2010-11-30", y=1157324.66, text="Nov 2010: £1.17M", showarrow=True, arrowhead=1)
fig.add_annotation(x="2011-11-30", y=1137634.0, text="Nov 2011: £1.16M", showarrow=True, arrowhead=1)
fig.add_annotation(x="2011-12-31", y=monthly_revenue.iloc[-1]['Revenue'], text="Partial month (9 days only)", showarrow=True, arrowhead=1)

# Highlight partial month marker
fig.add_scatter(x=[monthly_revenue.iloc[-1]['InvoiceDate']], y=[monthly_revenue.iloc[-1]['Revenue']], mode='markers', marker=dict(color='red', size=12), name='Partial Month')

fig.show()

**Commercial Insight:**
This trend line reveals the business's "heartbeat." We see strong seasonality (likely peaking in Q4 for the holidays). The **Demand Forecasting** system must account for these cycles to prevent over-purchasing in January and stock-outs in November. Any deviation from the previous year's trend without a clear marketing cause is a signal of market shifts or competitor pressure.

## 3. Average Order Value (AOV) Distribution
Understanding basket size and upsell opportunities.

In [4]:
# Standardize invoice column name if necessary
inv_col = 'Invoice' if 'Invoice' in df.columns else 'InvoiceNo'
aov_data = df.groupby(inv_col)['Revenue'].sum().reset_index()

# Removing outliers for better visualization (99th percentile)
upper_limit = aov_data['Revenue'].quantile(0.95)
aov_filtered = aov_data[aov_data['Revenue'] <= upper_limit]

fig = px.histogram(
    aov_filtered, 
    x='Revenue', 
    nbins=50, 
    title='Distribution of Order Value (up to 95th Percentile)',
    labels={'Revenue': 'Order Total (£)'},
    opacity=0.7
)
fig.add_vline(x=aov_data['Revenue'].mean(), line_dash="dash", line_color="red", 
              annotation_text=f"Mean: £{aov_data['Revenue'].mean():.2f}")
fig.show()

**Commercial Insight:**
Average Order Value (AOV) is a key lever for profitability. If the distribution is heavily skewed toward small orders, the shipping and handling costs are likely "bleeding your margin." This insight supports implementing thresholds for free shipping or bundle-based pricing strategies to shift the mean to the right.

## 4. Revenue Heatmap: Day of Week vs. Month
Optimizing marketing spend and operational capacity.

In [5]:
heatmap_data = df.pivot_table(index='DayOfWeek', columns='Month', values='Revenue', aggfunc='sum')
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig = px.imshow(
    heatmap_data, 
    labels=dict(x="Month", y="Day of Week", color="Revenue (£)"),
    x=months[:heatmap_data.shape[1]],
    y=[days[i] for i in heatmap_data.index],
    title="Revenue Heatmap: When Do We Make Money?",
    color_continuous_scale='YlGnBu'
)
fig.show()

**Commercial Insight:**
This heatmap tells the marketing team exactly when to push ad spend. High-revenue "hotspots" indicate peak purchase intent. Conversely, "cold" areas represent opportunities for midweek promotions to smooth out demand, making the warehouse and logistics operations more efficient and predictable.

## 5. Category Performance Analysis


In [6]:
cat_stats = df.groupby('Category').agg({
    'Revenue': 'sum',
    'StockCode': 'nunique'
}).rename(columns={'StockCode': 'SKUs'}).reset_index()
cat_stats['AvgRevenuePerSKU'] = cat_stats['Revenue'] / cat_stats['SKUs']
cat_stats = cat_stats.sort_values('Revenue', ascending=False)

fig = px.bar(
    cat_stats, 
    x='Revenue', 
    y='Category', 
    orientation='h',
    title='Category Revenue Performance',
    hover_data=['SKUs', 'AvgRevenuePerSKU'],
    labels={'Revenue': 'Total Revenue (£)', 'Category': 'Category'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

Highest value category by revenue-per-SKU is Doormats.

**Commercial Insight:**
Understanding category dominance allows for high-level resource allocation. If "JUMBO" or "SET" items dominate revenue, the commercial strategy should focus on bulk-shipping optimization and kitting. This also helps identify if you are over-reliant on a single category, which increases risk if that market segment fluctuates.

## 6. Price Variance: Top 10 SKUs with High Instability
Identifying inconsistent pricing or aggressive promotional cycles.

In [7]:
admin_codes_var = ['M', 'POST', 'DOT', 'ADJUST', 'ADJUST2']
df_var = df[~df['StockCode'].isin(admin_codes_var) & ~df['StockCode'].isna()]
price_stats = df_var.groupby(['StockCode', 'Description'])['Price'].agg(['std', 'mean']).reset_index()
price_stats['COV'] = price_stats['std'] / price_stats['mean']
top_variance = price_stats.dropna().sort_values('COV', ascending=False).head(10)

fig = px.bar(
    top_variance, 
    x='COV', 
    y='Description', 
    orientation='h',
    title='Top 10 SKUs by Price Variance (Coefficient of Variation)',
    labels={'COV': 'Coefficient of Variation (std/mean)'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

A product whose price changes frequently may indicate inconsistent pricing strategy, not a deliberate elasticity play.

**Commercial Insight:**
High price variance signals either inconsistent discounting or a lack of a unified pricing strategy. If the same product is sold at wildly different prices within the same market, you are likely leaving money on the table or confusing your customers. These SKUs are the prime targets for **Price Elasticity** analysis to find the "sweet spot" that maximizes margin without destroying volume.

## 7. Customer Type Revenue Split


In [ ]:
customer_split = df.set_index(pd.to_datetime(df['InvoiceDate'])).groupby(['customer_type', pd.Grouper(freq='ME')])['Revenue'].sum().reset_index()
fig = px.bar(
    customer_split, 
    x='InvoiceDate', 
    y='Revenue', 
    color='customer_type',
    title='Monthly Revenue Split: Wholesale vs Retail',
    barmode='stack',
    labels={'Revenue': 'Revenue (£)', 'InvoiceDate': 'Month', 'customer_type': 'Customer Type'}
)
fig.show()

Wholesale buyers respond differently to price changes than retail customers.

In [ ]:
# Dynamic Summary Calculations
total_tx = len(df)
top_prod_info = df.groupby(['StockCode', 'Description'])['Revenue'].sum()
top_prod_name = top_prod_info.idxmax()[1]
top_prod_rev = top_prod_info.max()
cust_split = df['customer_type'].value_counts(normalize=True) * 100

# Calculate Highest Value Category by Revenue-per-SKU (matching Fix 4 logic)
cat_stats = df.groupby('Category').agg({
    'Revenue': 'sum',
    'StockCode': 'nunique'
})
cat_stats['RevPerSKU'] = cat_stats['Revenue'] / cat_stats['StockCode']
highest_val_cat = cat_stats['RevPerSKU'].idxmax()

print(f"Total Transactions: {total_tx:,}")
print(f"Top Product: {top_prod_name} (£{top_prod_rev:,.2f})")
print(f"Wholesale: {cust_split.get('WHOLESALE', 0):.1f}%")
print(f"Retail: {cust_split.get('RETAIL', 0):.1f}%")
print(f"Highest Value Category: {highest_val_cat}")

## EDA Summary: Key Commercial Findings

- **Total clean transactions:** 776,842
- **Revenue generating period:** Dec 2009 to Nov 2011 (Dec 2011 partial)
- **Top product by revenue:** REGENCY CAKESTAND 3 TIER at £277,656.25
- **Highest value category by revenue-per-SKU:** Doormats
- **Customer base:** 63.5% wholesale, 36.5% retail by transaction count